# Chunking Strategy Evaluation

This notebook evaluates the performance of different chunking strategies:
- **baseline**: Simple character-based chunking
- **sentence**: Sentence-aware chunking  
- **adaptive**: Dynamic strategy selection based on document type

All plots will be displayed inline for easy viewing and analysis.

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Add parent directory to path for imports
sys.path.insert(0, str(Path.cwd().parent))

import json
import os
from typing import List, Dict, Optional, Any
from collections import defaultdict
import pandas as pd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Import project modules
from retriever import Retriever, detect_namespace
from prompt_builder import PromptBuilder
from gemini_integration import init_gemini, call_gemini, load_api_keys
from utils import DEFAULT_API_KEYS_PATH, DEFAULT_TOP_K

import google.generativeai as genai

# Configure matplotlib for inline plots
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Configuration
STRATEGIES = ["baseline", "sentence", "adaptive"]
TOP_K = 5
API_KEYS_PATH = DEFAULT_API_KEYS_PATH
QUERIES_FILE = "evaluation_queries.json"  # Relative to evaluation folder
OUTPUT_DIR = Path("results")  # Results will be saved here

print("✅ Imports loaded successfully!")
print(f"📁 Working directory: {Path.cwd()}")
print(f"📊 Strategies to evaluate: {', '.join(STRATEGIES)}")


## 3. Load Evaluation Queries

In [ ]:
# Load queries
queries_path = Path(QUERIES_FILE)
if queries_path.exists():
    with open(queries_path, "r", encoding="utf-8") as f:
        queries = json.load(f)
    print(f"✅ Loaded {len(queries)} queries from {queries_path}")
else:
    print(f"⚠️ Query file not found: {queries_path}")
    print("Please ensure evaluation_queries.json exists in the evaluation folder")
    queries = []

print(f"\n📝 Sample queries:")
for i, q in enumerate(queries[:3], 1):
    print(f"  {i}. {q.get('query', 'N/A')[:50]}...")

## 4. Helper Functions

In [ ]:
def evaluate_retrieval(retriever, query, strategy, top_k=5, expected_namespace=None):
    """Evaluate retrieval for a single query and strategy."""
    import sys
    from io import StringIO
    old_stdout = sys.stdout
    sys.stdout = StringIO()
    
    try:
        chunks = retriever.retrieve(
            query=query, top_k=top_k, strategy=strategy, include_metadata=True
        )
    finally:
        sys.stdout = old_stdout
    
    if not chunks:
        return {
            "avg_score": 0.0, "max_score": 0.0, "min_score": 0.0,
            "num_results": 0, "detected_namespace": "none",
            "namespace_correct": False, "doc_types": {}, "namespaces": {},
            "unique_docs": 0, "chunks": [],
        }
    
    scores = [chunk["score"] for chunk in chunks]
    namespaces = [chunk.get("namespace", "unknown") for chunk in chunks]
    detected_namespace = namespaces[0] if namespaces else "none"
    
    doc_types = defaultdict(int)
    for chunk in chunks:
        doc_type = chunk.get("metadata", {}).get("doc_type", "unknown")
        if isinstance(doc_type, str):
            doc_types[doc_type] += 1
    
    namespace_dist = defaultdict(int)
    for ns in namespaces:
        namespace_dist[ns] += 1
    
    unique_docs = len(set(chunk.get("doc_id", "") for chunk in chunks))
    
    namespace_correct = False
    if expected_namespace:
        namespace_correct = detected_namespace == expected_namespace
    
    return {
        "avg_score": np.mean(scores), "max_score": np.max(scores),
        "min_score": np.min(scores), "std_score": np.std(scores),
        "num_results": len(chunks), "detected_namespace": detected_namespace,
        "namespace_correct": namespace_correct, "expected_namespace": expected_namespace,
        "doc_types": dict(doc_types), "namespaces": dict(namespace_dist),
        "unique_docs": unique_docs, "chunks": chunks,
    }


def recall_at_k(chunks, gold_keywords):
    """Compute recall@K: whether any gold keyword appears in retrieved chunks."""
    if not chunks or not gold_keywords:
        return 0.0
    text = " ".join([chunk.get("chunk_text_only", chunk.get("text", "")) for chunk in chunks])
    return 1.0 if any(keyword in text for keyword in gold_keywords) else 0.0


def precision_at_k(chunks, gold_keywords):
    """Compute precision@K: fraction of chunks containing at least one gold keyword."""
    if not chunks or not gold_keywords:
        return 0.0
    total_hits = sum(
        any(keyword in chunk.get("chunk_text_only", chunk.get("text", "")) for keyword in gold_keywords)
        for chunk in chunks
    )
    return total_hits / len(chunks) if chunks else 0.0


def llm_grade_answer(question, answer, gold_keywords, gemini_model):
    """
    Use Gemini to grade the correctness of an answer.
    Returns a score between 0 and 1, or None if grading fails.
    """
    if not gold_keywords:
        return None  # Skip if no gold keywords provided
    
    prompt = f"""הערך את איכות התשובה הבאה בעברית.

שאלה: {question}
תשובה: {answer}

התשובה צריכה להכיל את המושגים הבאים (לא בהכרח התאמה מדויקת): {', '.join(gold_keywords)}.

החזר ציון בין 0 ל-1 בלבד (מספר עשרוני)."""
    
    try:
        response = gemini_model.generate_content(prompt)
        score_text = response.text.strip()
        # Extract number from response
        import re
        numbers = re.findall(r'\d+\.?\d*', score_text)
        if numbers:
            score = float(numbers[0])
            return max(0.0, min(1.0, score))
    except Exception as e:
        print(f"[WARN] LLM grading failed: {e}")
    
    return None


print("✅ Helper functions defined")

## 5. Initialize Retriever

In [ ]:
# Initialize retriever
print("[STEP] Initializing retriever...")
retriever = Retriever(api_keys_path=API_KEYS_PATH)
print("[OK] Retriever ready!")

# Initialize Gemini for answer generation and grading (optional)
ENABLE_ANSWER_EVALUATION = True  # Set to False to skip answer generation/grading
if ENABLE_ANSWER_EVALUATION:
    try:
        api_keys = load_api_keys(API_KEYS_PATH)
        gemini_model = init_gemini(api_keys, "gemini-2.5-flash")
        prompt_builder = PromptBuilder()
        print("[OK] Gemini model ready for answer evaluation!")
    except Exception as e:
        print(f"[WARN] Could not initialize Gemini: {e}")
        print("[INFO] Answer evaluation will be skipped")
        ENABLE_ANSWER_EVALUATION = False
        gemini_model = None
        prompt_builder = None
else:
    gemini_model = None
    prompt_builder = None

## 6. Run Evaluation

In [ ]:
# Run evaluation across all strategies
results = []

print(f"\n[EVALUATION] Testing {len(queries)} queries across {len(STRATEGIES)} strategies...")
if ENABLE_ANSWER_EVALUATION:
    print("[INFO] Answer generation and LLM grading enabled")
print("=" * 70)

for query_info in tqdm(queries, desc="Processing queries"):
    query = query_info["query"]
    expected_namespace = query_info.get("expected_namespace")
    category = query_info.get("category", "general")
    gold_keywords = query_info.get("gold_keywords", [])  # Optional: for precision/recall
    
    for strategy in STRATEGIES:
        # 1. Evaluate retrieval
        metrics = evaluate_retrieval(
            retriever=retriever,
            query=query,
            strategy=strategy,
            top_k=TOP_K,
            expected_namespace=expected_namespace,
        )
        
        # 2. Compute precision/recall if gold keywords provided
        recall = recall_at_k(metrics["chunks"], gold_keywords) if gold_keywords else None
        precision = precision_at_k(metrics["chunks"], gold_keywords) if gold_keywords else None
        
        # 3. Generate and evaluate answer if enabled
        answer = None
        llm_score = None
        if ENABLE_ANSWER_EVALUATION and metrics["chunks"]:
            try:
                # Generate answer using prompt builder
                prompt = prompt_builder.build_prompt(
                    question=query,
                    chunks=metrics["chunks"],
                    include_sources=True,
                )
                answer = call_gemini(gemini_model, prompt)
                
                # Grade answer if gold keywords provided
                if gold_keywords:
                    llm_score = llm_grade_answer(query, answer, gold_keywords, gemini_model)
            except Exception as e:
                print(f"[WARN] Answer generation failed for {query[:30]}...: {e}")
        
        # Store results
        result = {
            "query": query,
            "category": category,
            "strategy": strategy,
            "expected_namespace": expected_namespace,
            "detected_namespace": metrics["detected_namespace"],
            "namespace_correct": metrics["namespace_correct"],
            "avg_score": metrics["avg_score"],
            "max_score": metrics["max_score"],
            "min_score": metrics["min_score"],
            "std_score": metrics["std_score"],
            "num_results": metrics["num_results"],
            "unique_docs": metrics["unique_docs"],
            "doc_types": json.dumps(metrics["doc_types"], ensure_ascii=False),
            "namespaces": json.dumps(metrics["namespaces"], ensure_ascii=False),
        }
        
        # Add optional metrics
        if recall is not None:
            result["recall_at_k"] = recall
        if precision is not None:
            result["precision_at_k"] = precision
        if answer:
            result["answer"] = answer
        if llm_score is not None:
            result["llm_score"] = llm_score
        
        results.append(result)

# Create DataFrame
df = pd.DataFrame(results)
print(f"\n✅ Evaluation complete! {len(df)} results collected.")

## 7. Save Results

In [ ]:
# Ensure OUTPUT_DIR is defined
if 'OUTPUT_DIR' not in globals():
    from pathlib import Path
    OUTPUT_DIR = Path("results")
    OUTPUT_DIR.mkdir(exist_ok=True)

# Save results
df.to_csv(OUTPUT_DIR / "evaluation_results.csv", index=False, encoding="utf-8")
print(f"✅ Saved results to {OUTPUT_DIR / 'evaluation_results.csv'}")

# Display summary
print(f"\n📊 Summary:")
print(f"  Total queries: {df['query'].nunique()}")
print(f"  Total tests: {len(df)}")
print(f"  Strategies: {', '.join(df['strategy'].unique())}")

## 8. Compute Statistics

In [ ]:
# Compute strategy statistics
agg_dict = {
    "avg_score": ["mean", "std", "min", "max"],
    "max_score": ["mean", "std"],
    "namespace_correct": "mean",
    "num_results": "mean",
    "unique_docs": "mean",
}

# Add optional metrics if they exist
if "recall_at_k" in df.columns:
    agg_dict["recall_at_k"] = "mean"
if "precision_at_k" in df.columns:
    agg_dict["precision_at_k"] = "mean"
if "llm_score" in df.columns:
    agg_dict["llm_score"] = ["mean", "std"]

strategy_stats = df.groupby("strategy").agg(agg_dict).round(4)
strategy_stats.columns = ["_".join(col).strip() for col in strategy_stats.columns]
strategy_stats = strategy_stats.reset_index()

# Compute namespace accuracy
namespace_stats = df.groupby(["expected_namespace", "strategy"]).agg({
    "namespace_correct": ["mean", "count"],
}).round(4)
namespace_stats.columns = ["accuracy", "count"]
namespace_stats = namespace_stats.reset_index()

# Display statistics
print("📈 Strategy Statistics:")
print(strategy_stats.to_string(index=False))
strategy_stats.to_csv(OUTPUT_DIR / "strategy_statistics.csv", index=False, encoding="utf-8")
namespace_stats.to_csv(OUTPUT_DIR / "namespace_statistics.csv", index=False, encoding="utf-8")

## 9. Visualizations - Strategy Comparison

In [ ]:
# Strategy comparison plot
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Average score comparison
ax = axes[0, 0]
strategy_means = df.groupby("strategy")["avg_score"].mean().sort_values(ascending=False)
bars = ax.bar(strategy_means.index, strategy_means.values, 
              color=["#1f77b4", "#ff7f0e", "#2ca02c"])
ax.set_ylabel("Average Retrieval Score", fontsize=11)
ax.set_xlabel("Chunking Strategy", fontsize=11)
ax.set_title("Average Retrieval Score by Strategy", fontsize=12, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
for i, (strategy, val) in enumerate(strategy_means.items()):
    ax.text(i, val + 0.01, f"{val:.3f}", ha="center", va="bottom", fontsize=9)

# Score distribution
ax = axes[0, 1]
for strategy in df["strategy"].unique():
    scores = df[df["strategy"] == strategy]["avg_score"]
    ax.hist(scores, alpha=0.6, label=strategy, bins=15)
ax.set_xlabel("Average Retrieval Score", fontsize=11)
ax.set_ylabel("Frequency", fontsize=11)
ax.set_title("Score Distribution by Strategy", fontsize=12, fontweight="bold")
ax.legend()
ax.grid(axis="y", alpha=0.3)

# Namespace accuracy
ax = axes[1, 0]
namespace_acc = df.groupby("strategy")["namespace_correct"].mean()
bars = ax.bar(namespace_acc.index, namespace_acc.values,
              color=["#1f77b4", "#ff7f0e", "#2ca02c"])
ax.set_ylabel("Namespace Detection Accuracy", fontsize=11)
ax.set_xlabel("Chunking Strategy", fontsize=11)
ax.set_title("Namespace Detection Accuracy by Strategy", fontsize=12, fontweight="bold")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)
for i, (strategy, val) in enumerate(namespace_acc.items()):
    ax.text(i, val + 0.02, f"{val:.2%}", ha="center", va="bottom", fontsize=9)

# Unique documents
ax = axes[1, 1]
unique_docs_means = df.groupby("strategy")["unique_docs"].mean()
bars = ax.bar(unique_docs_means.index, unique_docs_means.values,
              color=["#1f77b4", "#ff7f0e", "#2ca02c"])
ax.set_ylabel("Average Unique Documents", fontsize=11)
ax.set_xlabel("Chunking Strategy", fontsize=11)
ax.set_title("Document Diversity by Strategy", fontsize=12, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
for i, (strategy, val) in enumerate(unique_docs_means.items()):
    ax.text(i, val + 0.1, f"{val:.2f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "strategy_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Saved plot to {OUTPUT_DIR / 'strategy_comparison.png'}")

## 10. Visualization - Namespace Accuracy Heatmap

In [ ]:
# Namespace accuracy heatmap
pivot = namespace_stats.pivot(index="expected_namespace", columns="strategy", values="accuracy")

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pivot, annot=True, fmt=".2%", cmap="YlOrRd",
            cbar_kws={"label": "Accuracy"}, ax=ax)
ax.set_title("Namespace Detection Accuracy\n(Expected vs. Detected)", 
             fontsize=13, fontweight="bold", pad=15)
ax.set_xlabel("Chunking Strategy", fontsize=11)
ax.set_ylabel("Expected Namespace", fontsize=11)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "namespace_accuracy_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Saved plot to {OUTPUT_DIR / 'namespace_accuracy_heatmap.png'}")

## 11. Visualization - Category Analysis

In [ ]:
# Category analysis
category_strategy = df.groupby(["category", "strategy"])["avg_score"].mean().reset_index()
pivot = category_strategy.pivot(index="category", columns="strategy", values="avg_score")

fig, ax = plt.subplots(figsize=(12, 6))
pivot.plot(kind="bar", ax=ax, color=["#1f77b4", "#ff7f0e", "#2ca02c"], width=0.8)
ax.set_ylabel("Average Retrieval Score", fontsize=11)
ax.set_xlabel("Query Category", fontsize=11)
ax.set_title("Retrieval Performance by Query Category and Strategy", 
             fontsize=12, fontweight="bold")
ax.legend(title="Strategy", title_fontsize=10)
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "category_analysis.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Saved plot to {OUTPUT_DIR / 'category_analysis.png'}")

## 12. Summary and Conclusions

## 13. Answer Quality Analysis (if enabled)


In [ ]:
# Analyze answer quality if answers were generated
if "llm_score" in df.columns and "answer" in df.columns:
    print("📊 Answer Quality Analysis:")
    print("=" * 70)
    
    # LLM score by strategy
    llm_by_strategy = df.groupby("strategy")["llm_score"].agg(["mean", "std", "count"]).round(4)
    print("\nLLM Answer Quality Score by Strategy:")
    print(llm_by_strategy)
    
    # Precision/Recall if available
    if "precision_at_k" in df.columns and "recall_at_k" in df.columns:
        pr_by_strategy = df.groupby("strategy")[["precision_at_k", "recall_at_k"]].mean().round(4)
        print("\nPrecision and Recall by Strategy:")
        print(pr_by_strategy)
        
        # Plot precision/recall
        fig, ax = plt.subplots(figsize=(10, 6))
        pr_by_strategy.plot(kind="bar", ax=ax, color=["#1f77b4", "#ff7f0e"])
        ax.set_ylabel("Score", fontsize=11)
        ax.set_xlabel("Chunking Strategy", fontsize=11)
        ax.set_title("Precision and Recall by Strategy", fontsize=12, fontweight="bold")
        ax.legend(title="Metric")
        ax.grid(axis="y", alpha=0.3)
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / "precision_recall_by_strategy.png", dpi=300, bbox_inches="tight")
        plt.show()
        print(f"✅ Saved plot to {OUTPUT_DIR / 'precision_recall_by_strategy.png'}")
    
    # Plot LLM scores
    fig, ax = plt.subplots(figsize=(10, 6))
    llm_means = df.groupby("strategy")["llm_score"].mean().sort_values(ascending=False)
    bars = ax.bar(llm_means.index, llm_means.values, color=["#1f77b4", "#ff7f0e", "#2ca02c"])
    ax.set_ylabel("Average LLM Quality Score", fontsize=11)
    ax.set_xlabel("Chunking Strategy", fontsize=11)
    ax.set_title("Answer Quality by Strategy (LLM Grading)", fontsize=12, fontweight="bold")
    ax.set_ylim(0, 1)
    ax.grid(axis="y", alpha=0.3)
    for i, (strategy, val) in enumerate(llm_means.items()):
        ax.text(i, val + 0.02, f"{val:.3f}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "llm_score_by_strategy.png", dpi=300, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved plot to {OUTPUT_DIR / 'llm_score_by_strategy.png'}")
    
else:
    print("ℹ️ Answer evaluation was not enabled or no answers were generated.")
    print("   Set ENABLE_ANSWER_EVALUATION = True and provide gold_keywords in queries to enable.")


In [ ]:
# Find best strategy
best_strategy = strategy_stats.loc[strategy_stats["avg_score_mean"].idxmax(), "strategy"]
best_avg_score = strategy_stats["avg_score_mean"].max()
overall_namespace_acc = df["namespace_correct"].mean()

print("=" * 70)
print("EVALUATION SUMMARY")
print("=" * 70)
print(f"\n✅ Best Performing Strategy (by retrieval score): {best_strategy}")
print(f"   Average Retrieval Score: {best_avg_score:.4f}")
print(f"\n✅ Overall Namespace Detection Accuracy: {overall_namespace_acc:.2%}")

# Add answer quality summary if available
if "llm_score" in strategy_stats.columns:
    best_llm_strategy = strategy_stats.loc[strategy_stats["llm_score_mean"].idxmax(), "strategy"]
    best_llm_score = strategy_stats["llm_score_mean"].max()
    print(f"\n✅ Best Strategy (by answer quality): {best_llm_strategy}")
    print(f"   Average LLM Quality Score: {best_llm_score:.4f}")

if "precision_at_k" in strategy_stats.columns and "recall_at_k" in strategy_stats.columns:
    best_precision_strategy = strategy_stats.loc[strategy_stats["precision_at_k"].idxmax(), "strategy"]
    best_recall_strategy = strategy_stats.loc[strategy_stats["recall_at_k"].idxmax(), "strategy"]
    print(f"\n✅ Best Precision: {best_precision_strategy} ({strategy_stats.loc[strategy_stats['precision_at_k'].idxmax(), 'precision_at_k']:.4f})")
    print(f"✅ Best Recall: {best_recall_strategy} ({strategy_stats.loc[strategy_stats['recall_at_k'].idxmax(), 'recall_at_k']:.4f})")

print(f"\n✅ All results saved to: {OUTPUT_DIR}")
print("=" * 70)